# Project Master Runner

This is the central entry point for the **Ecommerce Sentiment Analysis** project. It imports the logic from internal notebooks and executes the analysis pipeline using TensorFlow Metal acceleration.

## 1. Data Exploration & Cleaning

In [1]:
%load_ext autoreload
%autoreload 2

# Import the DataExploration class from the notebooks folder
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('notebooks'))

from notebooks.CapstoneData import DataExploration

# Define the dataset path
data_path = 'Ecommerce_dataset/train_data.csv'

# Instantiate and run the analysis
eda = DataExploration(data_path)
eda.get_summary()
eda.get_sentiment_distribution()
eda.remove_nulls(columns=['reviews.text', 'sentiment'])

print(f"\nCleaned data ready with {len(eda.df)} rows.")

Dataset loaded successfully with 4001 rows and 8 columns.

DATASET SUMMARY

--- First 5 rows ---
                                                name   brand  \
0  All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi...  Amazon   
1        Amazon - Echo Plus w/ Built-In Hub - Silver  Amazon   
2  Amazon Echo Show Alexa-enabled Bluetooth Speak...  Amazon   
3  Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 ...  Amazon   
4  Brand New Amazon Kindle Fire 16gb 7" Ips Displ...  Amazon   

                                          categories  \
0  Electronics,iPad & Tablets,All Tablets,Fire Ta...   
1  Amazon Echo,Smart Home,Networking,Home & Tools...   
2  Amazon Echo,Virtual Assistant Speakers,Electro...   
3  eBook Readers,Fire Tablets,Electronics Feature...   
4  Computers/Tablets & Networking,Tablets & eBook...   

             primaryCategories              reviews.date  \
0                  Electronics  2016-12-26T00:00:00.000Z   
1         Electronics,Hardware  2018-01-17T00:00:00.000Z   
2

## 2. Model Training & Validation

Run the cell below to execute **`train_and_validate.py`** in-process: it trains all seven models on the cleaned `eda.df`, writes artifacts under **`deploy_models/`**, evaluates on **`Ecommerce_dataset/test_data_hidden.csv`**, and saves metrics plus charts under **`validation_outputs/`** (confusion matrices, macro P/R/F1, per-class heatmaps, accuracy bars, `metrics_summary.csv`).

Implementation lives in `train_and_validate.py` (`run_pipeline`). Terminal: `python train_and_validate.py`. To **only** re-run evaluation and plots from existing checkpoints: `python validate_deploy_models.py`.

> **Note:** BERT / DistilBERT / RoBERTa / DeBERTa / RNN use TensorFlow with Metal GPU when available.

In [3]:
import os
import sys

# Project root must be on sys.path and the working directory (train_and_validate pins both on import)
_REPO_ROOT = os.path.abspath(os.getcwd())
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import train_and_validate as tv

# Train on notebook EDA DataFrame; validate on hidden test set; save under deploy_models/
results = tv.run_pipeline(eda.df)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

--- Training Logistic Regression ---
DataFrame provided directly with 4001 rows.

Cleaning Complete: Removed 10 rows with null values.
New row count: 3991

DATASET SUMMARY

--- First 5 rows ---
                                                name   brand  \
0  All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi...  Amazon   
1        Amazon - Echo Plus w/ Built-In Hub - Silver  Amazon   
2  Amazon Echo Show Alexa-enabled Bluetooth Speak...  Amazon   
3  Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 ...  Amazon   
4  Brand New Amazon Kindle Fire 16gb 7" Ips Displ...  Amazon   

                                          categories  \
0  Electronics,iPad & Tablets,All Tablets,Fire Ta...   
1  Amazon Echo,Smart Home,Networking,Home & Tools...   
2  Amazon Echo,Virtual Assistant Speakers,Electro...   
3  eBook Readers,Fire Tablets,Electronics Feature...   
4  Computers/Tablets & Networking,Tablets & eBook...   



Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_transform.weight', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'cla

Starting training for 3 epochs...
Epoch 1/3
283/283 [==============================] - 94s 289ms/step - loss: 0.3143 - accuracy: 0.9290 - val_loss: 0.1587 - val_accuracy: 0.9449
Epoch 2/3
283/283 [==============================] - 72s 256ms/step - loss: 0.1373 - accuracy: 0.9584 - val_loss: 0.1270 - val_accuracy: 0.9549
Epoch 3/3
50/50 [==============================] - 7s 140ms/step - loss: 0.1289 - accuracy: 0.9566
Final Validation Accuracy: 0.9566
Model saved to /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_distilbert_distilbert_tf

--- Training BERT ---
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
DataFrame provided directly with 4001 rows.

Cleaning Complete: Removed 10 rows with null values.
New row count: 3991

DATASET SUMMARY

--- First 5 rows ---
                                                name   brand  \
0  All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi...  Amazon   
1        Amazon - Echo 

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT model loaded!
Starting training...
Epoch 1/3
266/266 [==============================] - 186s 595ms/step - loss: 0.3162 - accuracy: 0.9164 - val_loss: 0.1490 - val_accuracy: 0.9499
Epoch 2/3
266/266 [==============================] - 140s 526ms/step - loss: 0.1429 - accuracy: 0.9514 - val_loss: 0.1193 - val_accuracy: 0.9549
Epoch 3/3
266/266 [==============================] - 151s 566ms/step - loss: 0.0931 - accuracy: 0.9749 - val_loss: 0.1171 - val_accuracy: 0.9612
Evaluating...
67/67 [==============================] - 18s 263ms/step - loss: 0.1171 - accuracy: 0.9612
Test Accuracy: 0.9612
Model and metadata saved to /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT_bert_tf and /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT.pkl

--- Fine-tuning RoBERTa (siebert backbone, 3-class head) ---
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
DataFrame pro

...

Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT_bert_tf
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_roberta_hf_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_roberta_hf_tf
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


All model checkpoint layers were used when initializing TFDebertaV2ForSequenceClassification.

All the layers of TFDebertaV2ForSequenceClassification were initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDebertaV2ForSequenceClassification for predictions without further training.
The tokenizer you are loading from '/Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf

[Logistic]
              precision    recall  f1-score   support

    Negative       0.06      0.88      0.11        24
     Neutral       0.24      0.10      0.14        39
    Positive       0.98      0.64      0.77       934

    accuracy                           0.62       997
   macro avg       0.42      0.54      0.34       997
weighted avg       0.93      0.62      0.73       997

Accuracy: 0.6228686058174524

[SVM]
              precision    recall  f1-score   support

    Negative       1.00      0.29      0.45        24
     Neutral       0.91      0.26      0.40        39
    Positive       0.95      1.00      0.98       934

    accuracy                           0.95       997
   macro avg       0.95      0.52      0.61       997
weighted avg       0.95      0.95      0.94       997

Accuracy: 0.9538615847542627

[RNN]
              precisi

## 3. Sentiment Models Initialization & Comparison

**After Section 2 finishes** (training + validation on the hidden test set), artifacts live under `deploy_models/`. Run this cell to load every saved model into the `models` dict for Sections 4–5.

If you trained from a terminal instead (`python train_and_validate.py`), run Section 2’s imports once from the repo root, or `os.chdir` to the project root before this cell so paths resolve.

In [4]:
import os
import sys

sys.path.append(os.path.abspath('notebooks'))

from notebooks.sentiment_analysis_distilbert import SentimentAnalysisDistilBERT
from notebooks.sentiment_analysis_BERT import SentimentAnalysisBERT
from notebooks.sentiment_analysis_RNN import SentimentAnalysisRNN
from notebooks.sentiment_analysis_Logistic import SentimentAnalysis
from notebooks.sentiment_analysis_SVM import SentimentAnalysisSVM
from notebooks.sentiment_analysis_Roberta import SentimentAnalysisHF
from notebooks.sentiment_analysis_DeBERTa import SentimentAnalysisDeBERTa

# Metadata .pkl paths (see train_and_validate.DEPLOY_DIR)
_dm = os.path.join('deploy_models')
paths = {
    'DistilBERT': os.path.join(_dm, 'sentiment_model_distilbert.pkl'),
    'BERT':       os.path.join(_dm, 'sentiment_model_BERT.pkl'),
    'RNN':        os.path.join(_dm, 'sentiment_model_RNN.pkl'),
    'Logistic':   os.path.join(_dm, 'sentiment_model_TFD_LR.pkl'),
    'SVM':        os.path.join(_dm, 'sentiment_model_SVM.pkl'),
    'RoBERTa':    os.path.join(_dm, 'sentiment_model_roberta.pkl'),
    'DeBERTa':    os.path.join(_dm, 'sentiment_model_deberta.pkl'),
}

models = {}

for name, path in paths.items():
    if os.path.exists(path):
        print(f"Loading {name}...")
        if name == 'DistilBERT': models[name] = SentimentAnalysisDistilBERT(model_path=path)
        elif name == 'BERT':     models[name] = SentimentAnalysisBERT(model_path=path)
        elif name == 'RNN':      models[name] = SentimentAnalysisRNN(model_path=path)
        elif name == 'Logistic': models[name] = SentimentAnalysis(model_path=path)
        elif name == 'SVM':      models[name] = SentimentAnalysisSVM(model_path=path)
        elif name == 'RoBERTa':  models[name] = SentimentAnalysisHF(model_path=path)
        elif name == 'DeBERTa':  models[name] = SentimentAnalysisDeBERTa(model_path=path)
    else:
        print(f"❌ {name} not found at {path}")

print(f"\nInitialized {len(models)} models: {list(models.keys())}")

Loading DistilBERT...
Using GPU (Metal acceleration): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Some layers from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_distilbert_distilbert_tf were not used when initializing TFDistilBertForSequenceClassification: ['dropout_151']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_distilbert_distilbert_tf and are newly init

Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_distilbert_distilbert_tf
Loading BERT...
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Some layers from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT_bert_tf were not used when initializing TFBertForSequenceClassification: ['dropout_189']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT_bert_tf.
If your task is similar to the task the model of the checkpoint wa

Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_BERT_bert_tf
Loading RNN...
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Model loaded from deploy_models/sentiment_model_RNN.keras
Loading Logistic...
Model loaded from deploy_models/sentiment_model_TFD_LR.pkl
Loading SVM...
Model loaded from deploy_models/sentiment_model_SVM.pkl
Loading RoBERTa...
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_roberta_hf_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_roberta_hf_tf
Loading DeBERTa...
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


All model checkpoint layers were used when initializing TFDebertaV2ForSequenceClassification.

All the layers of TFDebertaV2ForSequenceClassification were initialized from the model checkpoint at /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDebertaV2ForSequenceClassification for predictions without further training.
The tokenizer you are loading from '/Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model loaded from /Users/rohanghosh/Desktop/PlayGround/Ecommerce-Sentiment-Analysis/deploy_models/sentiment_model_deberta_deberta_tf

Initialized 7 models: ['DistilBERT', 'BERT', 'RNN', 'Logistic', 'SVM', 'RoBERTa', 'DeBERTa']


## 4. Sample Prediction

Testing the models with a sample review.

In [8]:
sample_text = "This product is absolutely worst, I love the way i wasted my money!"
sample_title = "Great way how to waste money"

if not models:
    print("No models loaded. Please train them in Section 2 first.")
else:
    print(f"Review: {sample_text}\n")
    for name, model in models.items():
        try:
            prediction = model.predict(sample_text, sample_title)
            print(f"[{name:10}] Prediction: {prediction}")
        except Exception as e:
            print(f"[{name:10}] Error: {e}")

Review: This product is absolutely worst, I love the way i wasted my money!

[DistilBERT] Prediction: Positive
[BERT      ] Prediction: Positive
[RNN       ] Prediction: Positive
[Logistic  ] Prediction: Positive
Prediction confidence: 0.9731
[SVM       ] Prediction: Positive
[RoBERTa   ] Prediction: Positive
[DeBERTa   ] Prediction: Positive


## 5. Visual Analytics Report

Generating detailed reports using the SentimentReporter.

In [10]:
from notebooks.sentiment_reporter import SentimentReporter
reporter = SentimentReporter(output_dir='personal_update/outputs')

data_path = 'Ecommerce_dataset/test_data_hidden.csv'

# Instantiate and run the analysis
validation = DataExploration(data_path)

print("Generating Visual Analytics Report...")
reporter.generate_visualizations(validation.df)

print("\n✅ PIPELINE COMPLETE")
print("Reports saved in 'personal_update/outputs/'")

Dataset loaded successfully with 1000 rows and 8 columns.
Generating Visual Analytics Report...
Generating Section 14: Overall Sentiment Plots...
Generating Section 15: Word Clouds...
Generating Section 16: Aspect Heatmap...

✅ PIPELINE COMPLETE
Reports saved in 'personal_update/outputs/'
